In [4]:
# ============================================================
# REVIEWER 1 / PROBLEM 1
# MORPHOLOGICAL REPRESENTATIVENESS AUDIT
# ONE-CELL GOOGLE COLAB CODE
#
# PURPOSE
# ------------------------------------------------------------
# 1. Reproduce the manuscript's strict-fair pairing:
#       Clean records              = 14,991
#       Segmented records          = 14,998
#       Clean unique keys          = 14,689
#       Segmented unique keys      = 14,696
#       Strict-fair paired subset  = 6,759
#       Excluded Clean unique      = 7,930
#
# 2. Re-apply the deterministic Relational CSE segmenter to
#    the Clean questions.
#
# 3. Quantitatively compare morphological complexity between:
#       A) Full Clean corpus
#       B) Full Clean unique-question universe
#       C) Strict-fair paired subset
#       D) Excluded Clean unique questions
#
# 4. Statistical comparison:
#       Strict-fair paired (N=6759)
#       vs
#       Excluded Clean unique (N=7930)
#
#    using:
#       - Welch's independent-samples t-test
#       - 95% CI for the mean difference
#       - Hedges' g
#       - Holm correction
#
# IMPORTANT
# ------------------------------------------------------------
# "CSE boundary" = a Relational-CSE-detected @@ morpheme
# boundary. It is an operational morphology measure, not
# a manually gold-annotated linguistic affix count.
# ============================================================


# ============================================================
# 0. IMPORTS / DEPENDENCIES
# ============================================================

import os
import sys
import re
import json
import glob
import math
import subprocess
import zipfile

from pathlib import Path
from functools import lru_cache
from typing import Any, Dict, List


def ensure(package, import_name=None):
    import_name = import_name or package
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package]
        )


ensure("xlrd==2.0.1", "xlrd")
ensure("openpyxl", "openpyxl")
ensure("numpy", "numpy")
ensure("pandas", "pandas")
ensure("scipy", "scipy")

import xlrd
import openpyxl
import numpy as np
import pandas as pd

from scipy import stats


try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False


pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option(
    "display.float_format",
    lambda x: f"{x:.6f}"
)


# ============================================================
# 1. FIND SOURCE FILES
# ============================================================

PATTERNS = {

    "base": [
        "baseline_15000*.json",
        "baseline*15000*.json"
    ],

    "seg": [
        "kazakh_segmented_15000*.json",
        "kazakh*segmented*15000*.json"
    ],

    "stems": [
        "qaz_stems_unik_edit*.xlsx"
    ],

    "jurnaqs": [
        "qaz_jurnaks*.xls"
    ],

    "endings": [
        "qaz_endings_seg*.xls"
    ],

    "stop": [
        "stop_words*.txt",
        "stop*words*.txt"
    ],
}


def find_matches(patterns):

    hits = []

    for pattern in patterns:

        hits += glob.glob(
            f"/content/**/{pattern}",
            recursive=True
        )

        hits += glob.glob(
            f"./**/{pattern}",
            recursive=True
        )

    unique = []
    seen = set()

    for hit in hits:

        p = str(Path(hit).resolve())

        if p not in seen and Path(p).is_file():
            seen.add(p)
            unique.append(p)

    # Prefer /content files.
    unique.sort(
        key=lambda p: (
            0 if Path(p).parent == Path("/content") else 1,
            len(Path(p).parts),
            -Path(p).stat().st_mtime
        )
    )

    return unique


def resolve_required_files():

    resolved = {}
    missing = []

    for key, patterns in PATTERNS.items():

        matches = find_matches(patterns)

        if matches:
            resolved[key] = matches[0]
        else:
            missing.append(key)

    if missing and IN_COLAB:

        print("\n📤 Кейбір файлдар табылмады:")
        print(", ".join(missing))

        print("\nКелесі 6 файлды жүктеңіз:")
        print("1. baseline_15000.json")
        print("2. kazakh_segmented_15000.json")
        print("3. qaz_stems_unik_edit.xlsx")
        print("4. qaz_jurnaks.xls")
        print("5. qaz_endings_seg.xls")
        print("6. stop_words.txt")

        files.upload()

        resolved = {}
        missing = []

        for key, patterns in PATTERNS.items():

            matches = find_matches(patterns)

            if matches:
                resolved[key] = matches[0]
            else:
                missing.append(key)

    if missing:

        raise FileNotFoundError(
            "Required files missing: "
            + ", ".join(missing)
        )

    print("\n================ FILES USED ================")

    for key, value in resolved.items():
        print(f"{key:10s}: {value}")

    return resolved


PATHS = resolve_required_files()


# ============================================================
# 2. ROBUST JSON LOADER
#    Supports JSON array, JSONL and brace-scanned JSON
# ============================================================

def normalize_records(data: Any) -> List[Dict[str, str]]:

    if not isinstance(data, list):
        raise ValueError(
            "QA data must be a list after parsing."
        )

    result = []

    for item in data:

        if not isinstance(item, dict):
            continue

        q = (
            item.get("question")
            or item.get("instruction")
            or ""
        )

        a = (
            item.get("answer")
            or item.get("response")
            or ""
        )

        q = str(q).strip()
        a = str(a).strip()

        if q and a:
            result.append({
                "question": q,
                "answer": a
            })

    if not result:
        raise ValueError(
            "No valid question/answer records found."
        )

    return result


def load_qa_records(path: str):

    text = Path(path).read_text(
        encoding="utf-8",
        errors="ignore"
    ).strip()

    if not text:
        raise ValueError(
            f"Empty file: {path}"
        )

    # Standard JSON array
    if text[0] == "[":

        try:
            return normalize_records(
                json.loads(text)
            )
        except Exception:
            pass

    # JSONL
    lines = [
        line.strip().rstrip(",")
        for line in text.splitlines()
        if line.strip()
    ]

    if lines and lines[0].startswith("{"):

        records = []
        ok = True

        for line in lines:

            try:
                records.append(
                    json.loads(line)
                )
            except Exception:
                ok = False
                break

        if ok and records:
            return normalize_records(records)

    # Robust brace scan
    objects = []

    buffer = []
    depth = 0

    in_string = False
    escaped = False
    started = False

    for char in text:

        if not started:

            if char == "{":
                started = True
                depth = 1
                buffer = ["{"]

            continue

        buffer.append(char)

        if in_string:

            if escaped:
                escaped = False

            elif char == "\\":
                escaped = True

            elif char == '"':
                in_string = False

        else:

            if char == '"':
                in_string = True

            elif char == "{":
                depth += 1

            elif char == "}":

                depth -= 1

                if depth == 0:

                    object_text = "".join(buffer)

                    buffer = []
                    started = False

                    try:
                        objects.append(
                            json.loads(object_text)
                        )
                    except Exception:
                        pass

    if not objects:

        raise ValueError(
            f"JSON parsing failed: {path}"
        )

    return normalize_records(objects)


base_rows = load_qa_records(
    PATHS["base"]
)

seg_rows = load_qa_records(
    PATHS["seg"]
)


# ============================================================
# 3. EXACT STRICT-FAIR NORMALIZATION
#    SAME LOGIC AS STRICT-FAIR EXPERIMENT
# ============================================================

_punct_space_left = re.compile(
    r"\s+([.,!?;:%)\]\}])"
)

_punct_space_right = re.compile(
    r"([(\[\{])\s+"
)

_multi_space = re.compile(
    r"\s+"
)


def norm_space_punct(text: str):

    text = text.replace(
        " - ",
        "-"
    )

    text = _punct_space_left.sub(
        r"\1",
        text
    )

    text = _punct_space_right.sub(
        r"\1",
        text
    )

    text = _multi_space.sub(
        " ",
        text
    ).strip()

    return text


def clean_view(text: str):

    text = (
        ""
        if text is None
        else str(text)
    )

    text = text.replace(
        "@@ ",
        ""
    ).replace(
        "@@",
        ""
    )

    return norm_space_punct(text)


def morph_marker_view(text: str):

    text = (
        ""
        if text is None
        else str(text)
    )

    return norm_space_punct(text)


def first_by_clean_key(rows):

    result = {}

    for row in rows:

        key = clean_view(
            row["question"]
        )

        # EXACTLY like strict-fair code:
        # keep first duplicate only
        if key and key not in result:
            result[key] = row

    return result


base_map = first_by_clean_key(
    base_rows
)

seg_map = first_by_clean_key(
    seg_rows
)


common_keys = sorted(
    set(base_map.keys())
    &
    set(seg_map.keys())
)


excluded_keys = sorted(
    set(base_map.keys())
    -
    set(seg_map.keys())
)


print(
    "\n================ PAIRING AUDIT ================"
)

print(
    f"Clean records                  = {len(base_rows):,}"
)

print(
    f"Segmented records              = {len(seg_rows):,}"
)

print(
    f"Clean unique normalized keys   = {len(base_map):,}"
)

print(
    f"Segmented unique norm. keys    = {len(seg_map):,}"
)

print(
    f"Strict-fair common keys        = {len(common_keys):,}"
)

print(
    f"Excluded Clean unique keys     = {len(excluded_keys):,}"
)

print(
    f"Clean duplicate rows collapsed = "
    f"{len(base_rows)-len(base_map):,}"
)

print(
    f"Seg duplicate rows collapsed   = "
    f"{len(seg_rows)-len(seg_map):,}"
)


# ============================================================
# 4. CRITICAL PAIRING VALIDATION
# ============================================================

EXPECTED = {

    "base_records": 14991,
    "seg_records": 14998,

    "base_unique": 14689,
    "seg_unique": 14696,

    "common": 6759,
    "excluded": 7930,
}


ACTUAL = {

    "base_records": len(base_rows),
    "seg_records": len(seg_rows),

    "base_unique": len(base_map),
    "seg_unique": len(seg_map),

    "common": len(common_keys),
    "excluded": len(excluded_keys),
}


wrong = {

    key: (
        ACTUAL[key],
        expected
    )

    for key, expected in EXPECTED.items()

    if ACTUAL[key] != expected
}


if wrong:

    raise RuntimeError(

        "\nSTOP!\n"
        "Uploaded datasets do NOT reproduce the "
        "manuscript strict-fair pairing.\n\n"
        f"Mismatches (actual, expected):\n{wrong}\n\n"
        "Use the exact experimental data files."
    )


print(
    "\n✅ Pairing counts EXACTLY match the manuscript."
)


# ============================================================
# 5. LOAD RELATIONAL CSE RESOURCES
# ============================================================

def clean_cell(value):

    if value is None:
        return ""

    return (
        str(value)
        .replace("\ufeff", "")
        .strip()
    )


# ------------------------------------------------------------
# 5.1 STEMS
# Original segmentation implementation reads ALL rows
# of first column.
# ------------------------------------------------------------

workbook = openpyxl.load_workbook(

    PATHS["stems"],

    read_only=True,
    data_only=True
)


worksheet = workbook.active


stems = []


for row in worksheet.iter_rows(

    min_col=1,
    max_col=1,
    values_only=True
):

    value = clean_cell(
        row[0]
    )

    if value:
        stems.append(value)


workbook.close()


# ------------------------------------------------------------
# 5.2 DERIVATIONAL SUFFIXES / JURNAQS
# Original code skips first row as header
# ------------------------------------------------------------

jurnaq_workbook = xlrd.open_workbook(
    PATHS["jurnaqs"]
)

jurnaq_sheet = jurnaq_workbook.sheet_by_index(
    0
)


jurnaqs = []


for row_number in range(
    1,
    jurnaq_sheet.nrows
):

    value = clean_cell(
        jurnaq_sheet.cell(
            row_number,
            0
        ).value
    )

    if value:
        jurnaqs.append(value)


# ------------------------------------------------------------
# 5.3 INFLECTIONAL ENDINGS
# Original code skips first row as header
# ------------------------------------------------------------

ending_workbook = xlrd.open_workbook(
    PATHS["endings"]
)

ending_sheet = ending_workbook.sheet_by_index(
    0
)


endings = []
ending_segmentations = []


for row_number in range(
    1,
    ending_sheet.nrows
):

    ending = clean_cell(
        ending_sheet.cell(
            row_number,
            0
        ).value
    )

    segmentation = (

        clean_cell(
            ending_sheet.cell(
                row_number,
                1
            ).value
        )

        if ending_sheet.ncols > 1

        else ""
    )

    if ending:

        endings.append(
            ending
        )

        ending_segmentations.append(
            segmentation
        )


# ------------------------------------------------------------
# 5.4 STOP WORDS
# SAME ENCODING FALLBACK ORDER AS ORIGINAL NOTEBOOK
# ------------------------------------------------------------

def read_lines_with_fallback(

    path: str,

    encodings=(
        "utf-8",
        "utf-8-sig",
        "cp1251",
        "cp1252",
        "latin-1"
    )
):

    last_error = None

    for encoding in encodings:

        try:

            with open(
                path,
                "r",
                encoding=encoding,
                errors="strict"
            ) as file:

                lines = file.read().splitlines()

            lines = [

                line
                .replace("\ufeff", "")
                .strip()

                for line in lines
            ]

            lines = [
                line
                for line in lines
                if line
            ]

            print(
                f"✅ stop_words encoding={encoding} "
                f"| non-empty={len(lines)}"
            )

            return lines

        except UnicodeDecodeError as error:

            last_error = error


    with open(

        path,
        "r",
        encoding="utf-8",
        errors="replace"

    ) as file:

        lines = file.read().splitlines()


    lines = [

        line
        .replace("\ufeff", "")
        .strip()

        for line in lines

        if line.strip()
    ]


    print(
        "⚠️ stop_words loaded with utf-8 errors=replace"
    )


    return lines


stop_words = read_lines_with_fallback(
    PATHS["stop"]
)


# ============================================================
# 6. RESOURCE AUDIT
# ============================================================

print(
    "\n================ RESOURCE AUDIT ================"
)

print(
    f"Stem entries                    = {len(stems):,}"
)

print(
    f"Unique stem forms               = {len(set(stems)):,}"
)

print(
    f"Derivational-suffix entries     = {len(jurnaqs):,}"
)

print(
    f"Unique derivational suffixes    = {len(set(jurnaqs)):,}"
)

print(
    f"Ending→segmentation mappings    = {len(endings):,}"
)

print(
    f"Unique surface endings          = {len(set(endings)):,}"
)

print(
    f"Stop-word non-empty entries     = {len(stop_words):,}"
)

print(
    f"Unique stop-word values         = {len(set(stop_words)):,}"
)


EXPECTED_RESOURCES = {

    "stems": 103624,
    "stems_unique": 103623,

    "jurnaqs": 146,
    "jurnaqs_unique": 101,

    "endings": 3316,
    "endings_unique": 3030,

    "stop": 189,
    "stop_unique": 188,
}


ACTUAL_RESOURCES = {

    "stems": len(stems),
    "stems_unique": len(set(stems)),

    "jurnaqs": len(jurnaqs),
    "jurnaqs_unique": len(set(jurnaqs)),

    "endings": len(endings),
    "endings_unique": len(set(endings)),

    "stop": len(stop_words),
    "stop_unique": len(set(stop_words)),
}


resource_wrong = {

    key: (
        ACTUAL_RESOURCES[key],
        expected
    )

    for key, expected
    in EXPECTED_RESOURCES.items()

    if ACTUAL_RESOURCES[key] != expected
}


if resource_wrong:

    raise RuntimeError(

        "\nSTOP!\n"
        "Morphological resource counts do NOT match "
        "the manuscript.\n\n"
        f"Mismatches (actual, expected):\n"
        f"{resource_wrong}\n\n"
        "Check that the exact CSE resource files "
        "used in the experiment were uploaded."
    )


print(
    "\n✅ Resource counts EXACTLY match the manuscript."
)


# ============================================================
# 7. FAST LOOKUP STRUCTURES
#    They do NOT change the segmentation logic.
# ============================================================

stem_set = set(
    stems
)

jurnaq_set = set(
    jurnaqs
)

stop_set = set(
    stop_words
)


# Original code:
# endings.index(ending)
# therefore FIRST occurrence must be retained.

ending_first_seg = {}


for ending, segmentation in zip(
    endings,
    ending_segmentations
):

    if ending not in ending_first_seg:

        ending_first_seg[
            ending
        ] = segmentation


# ============================================================
# 8. RELATIONAL CSE SEGMENTATION
# ============================================================

@lru_cache(maxsize=None)
def ending_stem_fast(word: str):

    length = len(word)

    minimum_stem_length = 2


    if length <= minimum_stem_length:

        return (
            word,
            ""
        )


    # Full-word stem check first
    if word.lower() in stem_set:

        return (
            word,
            ""
        )


    # --------------------------------------------------------
    # INFLECTIONAL ENDINGS:
    # RIGHT-TO-LEFT LONGEST-FIRST MATCHING
    #
    # The candidate must leave at least a 2-char stem.
    # --------------------------------------------------------

    maximum_ending_length = (
        length
        -
        minimum_stem_length
    )


    for ending_length in range(

        maximum_ending_length,
        0,
        -1

    ):

        candidate = word[
            -ending_length:
        ]

        stem = word[
            :-ending_length
        ]


        if (

            candidate in ending_first_seg

            and

            stem.lower() in stem_set

        ):

            return (
                stem,
                candidate
            )


    return (
        word,
        ""
    )


@lru_cache(maxsize=None)
def first_jurnaq_fast(word: str):

    length = len(word)


    if length <= 2:

        return (
            word,
            ""
        )


    # --------------------------------------------------------
    # DERIVATIONAL SUFFIXES:
    # SHORTEST-FIRST ITERATIVE STRIPPING
    #
    # 1 -> 2 -> 3 -> 4 characters
    # --------------------------------------------------------

    for suffix_length in range(

        1,
        min(
            4,
            length
        ) + 1

    ):

        candidate = word[
            -suffix_length:
        ]

        stem = word[
            :-suffix_length
        ]


        if (

            candidate in jurnaq_set

            and

            stem.lower() in stem_set

        ):

            return (
                stem,
                candidate
            )


    return (
        word,
        ""
    )


@lru_cache(maxsize=None)
def segment_word_fast(word: str):

    # Identify stem and inflectional ending
    stem, ending = ending_stem_fast(word)

    if ending:
        ending_seg = ending_first_seg.get(
            ending,
            ""
        )
    else:
        ending_seg = ""

    # --------------------------------------------------------
    # DERIVATIONAL SUFFIXES:
    # SHORTEST-FIRST ITERATIVE STRIPPING
    # --------------------------------------------------------

    jurnaq_seg = ""

    if len(stem) > 2:

        # Safety bound only; this does not alter
        # the normal segmentation logic.
        for _ in range(max(1, len(stem))):

            remaining_stem, jurnaq = first_jurnaq_fast(
                stem
            )

            if not jurnaq:
                break

            # Because suffixes are stripped from the right edge,
            # each newly detected suffix is prepended so that
            # the original surface order is restored.
            if jurnaq_seg:
                jurnaq_seg = (
                    f"{jurnaq}@@ "
                    f"{jurnaq_seg}"
                )
            else:
                jurnaq_seg = jurnaq

            stem = remaining_stem

    # --------------------------------------------------------
    # RECONSTRUCTION WITH @@
    # --------------------------------------------------------

    if ending_seg and jurnaq_seg:

        return (
            f"{stem}@@ "
            f"{jurnaq_seg}"
            f"{ending_seg}"
        )

    elif ending_seg:

        return (
            f"{stem}@@ "
            f"{ending_seg}"
        )

    elif jurnaq_seg:

        return (
            f"{stem}@@ "
            f"{jurnaq_seg}"
        )

    return stem


TOKEN_RE = re.compile(
    r"\w+|[^\w\s]",
    flags=re.UNICODE
)


def morpho_segment_fast(text: str):

    tokens = TOKEN_RE.findall(
        str(text)
    )


    processed = []


    for token in tokens:

        if token.lower() in stop_set:

            processed.append(
                token
            )


        elif token.isalpha():

            processed.append(
                segment_word_fast(
                    token
                )
            )


        else:

            processed.append(
                token
            )


    return " ".join(
        processed
    )


# ============================================================
# 9. CRITICAL VALIDATION AGAINST STORED SEGMENTED QUESTIONS
# ============================================================

print(
    "\n================ SEGMENTER VALIDATION ================"
)


exact_match = 0
boundary_match = 0

mismatch_examples = []


for key in common_keys:

    clean_question = clean_view(
        base_map[key]["question"]
    )


    generated = morph_marker_view(

        morpho_segment_fast(
            clean_question
        )
    )


    stored = morph_marker_view(
        seg_map[key]["question"]
    )


    if generated == stored:

        exact_match += 1

    elif len(mismatch_examples) < 10:

        mismatch_examples.append(
            (
                clean_question,
                generated,
                stored
            )
        )


    if (

        generated.count("@@")
        ==
        stored.count("@@")

    ):

        boundary_match += 1


n_common = len(
    common_keys
)


exact_rate = (
    exact_match
    /
    n_common
)


boundary_rate = (
    boundary_match
    /
    n_common
)


print(
    f"Exact normalized segmentation match "
    f"= {exact_match:,}/{n_common:,} "
    f"= {exact_rate:.2%}"
)


print(
    f"@@ boundary-count match "
    f"= {boundary_match:,}/{n_common:,} "
    f"= {boundary_rate:.2%}"
)


if mismatch_examples:

    print(
        "\nFirst segmentation mismatches "
        "(audit only):"
    )


    for i, (
        question,
        generated,
        stored
    ) in enumerate(
        mismatch_examples[:5],
        1
    ):

        print(
            f"\n[{i}] CLEAN:"
        )

        print(
            question
        )

        print(
            "GENERATED:"
        )

        print(
            generated
        )

        print(
            "STORED:"
        )

        print(
            stored
        )


# Boundary count is the key morphology statistic.
# Require >=99% agreement before analysis.

if boundary_rate < 0.99:

    raise RuntimeError(

        "\nSTOP!\n"
        f"Generated-vs-stored boundary agreement "
        f"is only {boundary_rate:.2%}.\n\n"
        "Do NOT use the morphological complexity "
        "results until the exact segmentation "
        "implementation/resources are reconciled."
    )


print(
    "\n✅ CSE boundary behavior validated at ≥99%."
)


# ============================================================
# 10. QUESTION-LEVEL MORPHOLOGICAL COMPLEXITY FEATURES
# ============================================================

def question_features(question: str):

    question = clean_view(
        question
    )


    tokens = TOKEN_RE.findall(
        question
    )


    alphabetic_words = [

        token

        for token in tokens

        if token.isalpha()
    ]


    if not alphabetic_words:

        return {

            "cse_boundaries_per_question": 0.0,

            "cse_boundaries_per_word": 0.0,

            "complex_word_proportion": 0.0,

            "mean_word_length_chars": 0.0,

            "max_cse_boundaries_in_word": 0.0,

            "alphabetic_words_per_question": 0.0,
        }


    boundaries_per_word = []


    for word in alphabetic_words:


        if word.lower() in stop_set:

            segmented_word = word


        else:

            segmented_word = segment_word_fast(
                word
            )


        boundaries_per_word.append(

            segmented_word.count(
                "@@"
            )
        )


    total_boundaries = int(
        sum(
            boundaries_per_word
        )
    )


    complex_words = int(

        sum(

            1

            for value in boundaries_per_word

            if value >= 1
        )
    )


    return {

        # Total Relational-CSE-detected boundaries
        "cse_boundaries_per_question":
            float(
                total_boundaries
            ),


        # Controls for question length
        "cse_boundaries_per_word":
            float(
                total_boundaries
                /
                len(alphabetic_words)
            ),


        # Proportion of alphabetic words that
        # received >=1 CSE boundary
        "complex_word_proportion":
            float(
                complex_words
                /
                len(alphabetic_words)
            ),


        # Surface-form word length
        "mean_word_length_chars":
            float(
                np.mean(
                    [
                        len(word)
                        for word
                        in alphabetic_words
                    ]
                )
            ),


        # Morphological depth proxy
        "max_cse_boundaries_in_word":
            float(
                max(
                    boundaries_per_word
                )
            ),


        # Auxiliary question-length control
        "alphabetic_words_per_question":
            float(
                len(
                    alphabetic_words
                )
            ),
    }


# ============================================================
# 11. ANALYZE 14,689 UNIQUE CLEAN QUESTIONS
# ============================================================

print(
    "\nComputing morphological features "
    "for 14,689 unique Clean questions..."
)


common_set = set(
    common_keys
)


feature_rows = []


for index, key in enumerate(
    base_map.keys(),
    1
):

    features = question_features(
        key
    )


    features[
        "question_key"
    ] = key


    if key in common_set:

        features[
            "selection_group"
        ] = "strict_fair_paired"

    else:

        features[
            "selection_group"
        ] = "excluded_clean"


    feature_rows.append(
        features
    )


    if index % 2000 == 0:

        print(
            f"  processed "
            f"{index:,}/"
            f"{len(base_map):,}"
        )


unique_df = pd.DataFrame(
    feature_rows
)


paired_df = unique_df[

    unique_df[
        "selection_group"
    ]
    ==
    "strict_fair_paired"

].copy()


excluded_df = unique_df[

    unique_df[
        "selection_group"
    ]
    ==
    "excluded_clean"

].copy()


# ============================================================
# 12. FULL CLEAN CORPUS (14,991 RECORDS)
# ============================================================

feature_by_key = (

    unique_df
    .set_index(
        "question_key"
    )
    .to_dict(
        orient="index"
    )
)


all_clean_features = []


for row in base_rows:

    key = clean_view(
        row["question"]
    )


    values = {

        feature_name: value

        for feature_name, value

        in feature_by_key[
            key
        ].items()

        if feature_name
        != "selection_group"
    }


    values[
        "question_key"
    ] = key


    all_clean_features.append(
        values
    )


all_clean_df = pd.DataFrame(
    all_clean_features
)


# ============================================================
# 13. CRITICAL SIZE CHECK
# ============================================================

assert len(
    all_clean_df
) == 14991


assert len(
    unique_df
) == 14689


assert len(
    paired_df
) == 6759


assert len(
    excluded_df
) == 7930


print(
    "\n✅ Final group sizes:"
)

print(
    f"Full Clean records     = {len(all_clean_df):,}"
)

print(
    f"Full Clean unique      = {len(unique_df):,}"
)

print(
    f"Strict-fair paired     = {len(paired_df):,}"
)

print(
    f"Excluded Clean unique  = {len(excluded_df):,}"
)


# ============================================================
# 14. PRIMARY MORPHOLOGICAL COMPLEXITY FEATURES
# ============================================================

PRIMARY_METRICS = [

    (
        "cse_boundaries_per_question",
        "CSE boundaries/question"
    ),

    (
        "cse_boundaries_per_word",
        "CSE boundaries/alphabetic word"
    ),

    (
        "complex_word_proportion",
        "Morphologically complex-word proportion"
    ),

    (
        "mean_word_length_chars",
        "Mean alphabetic word length (chars)"
    ),

    (
        "max_cse_boundaries_in_word",
        "Max CSE boundaries in one word"
    ),
]


# Auxiliary question-length control:
AUX_METRICS = [

    (
        "alphabetic_words_per_question",
        "Alphabetic words/question (length control)"
    ),
]


# ============================================================
# 15. DESCRIPTIVE REPRESENTATIVENESS
# ============================================================

def mean_sd(
    dataframe,
    column
):

    values = dataframe[
        column
    ].to_numpy(
        dtype=float
    )


    return (

        float(
            np.mean(
                values
            )
        ),

        float(
            np.std(
                values,
                ddof=1
            )
        )
    )


descriptive_rows = []


for column, label in (

    PRIMARY_METRICS
    +
    AUX_METRICS

):


    full_mean, full_sd = mean_sd(
        all_clean_df,
        column
    )


    unique_mean, unique_sd = mean_sd(
        unique_df,
        column
    )


    paired_mean, paired_sd = mean_sd(
        paired_df,
        column
    )


    excluded_mean, excluded_sd = mean_sd(
        excluded_df,
        column
    )


    if abs(
        unique_mean
    ) < 1e-15:

        relative_difference = np.nan

    else:

        relative_difference = (

            100.0

            *
            (
                paired_mean
                -
                unique_mean
            )

            /
            unique_mean
        )


    descriptive_rows.append({

        "Feature":
            label,


        "Full Clean all N":
            len(
                all_clean_df
            ),

        "Full Clean all mean":
            full_mean,

        "Full Clean all SD":
            full_sd,


        "Full Clean unique N":
            len(
                unique_df
            ),

        "Full Clean unique mean":
            unique_mean,

        "Full Clean unique SD":
            unique_sd,


        "Strict-fair N":
            len(
                paired_df
            ),

        "Strict-fair mean":
            paired_mean,

        "Strict-fair SD":
            paired_sd,


        "Excluded unique N":
            len(
                excluded_df
            ),

        "Excluded unique mean":
            excluded_mean,

        "Excluded unique SD":
            excluded_sd,


        "Strict-fair vs Full-unique relative diff %":
            relative_difference,
    })


descriptive_table = pd.DataFrame(
    descriptive_rows
)


# ============================================================
# 16. STATISTICAL FUNCTIONS
# ============================================================

def welch_mean_difference_ci(
    x,
    y,
    alpha=0.05
):

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )


    n1 = len(x)
    n2 = len(y)


    mean1 = np.mean(x)
    mean2 = np.mean(y)


    var1 = np.var(
        x,
        ddof=1
    )

    var2 = np.var(
        y,
        ddof=1
    )


    difference = (
        mean1
        -
        mean2
    )


    se_squared = (

        var1
        /
        n1

        +

        var2
        /
        n2
    )


    if se_squared <= 0:

        return (

            difference,
            difference,
            difference,
            np.inf
        )


    se = math.sqrt(
        se_squared
    )


    denominator = (

        (
            (
                var1
                /
                n1
            )
            ** 2
        )
        /
        (
            n1 - 1
        )

        +

        (
            (
                var2
                /
                n2
            )
            ** 2
        )
        /
        (
            n2 - 1
        )
    )


    if denominator > 0:

        df = (

            se_squared
            ** 2

            /
            denominator
        )

    else:

        df = np.inf


    if np.isfinite(df):

        critical = stats.t.ppf(

            1
            -
            alpha / 2,

            df
        )

    else:

        critical = stats.norm.ppf(

            1
            -
            alpha / 2
        )


    ci_low = (

        difference
        -
        critical * se
    )


    ci_high = (

        difference
        +
        critical * se
    )


    return (

        float(
            difference
        ),

        float(
            ci_low
        ),

        float(
            ci_high
        ),

        float(
            df
        )
    )


def hedges_g(
    x,
    y
):

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )


    n1 = len(x)
    n2 = len(y)


    var1 = np.var(
        x,
        ddof=1
    )

    var2 = np.var(
        y,
        ddof=1
    )


    pooled_variance = (

        (
            n1 - 1
        )
        *
        var1

        +

        (
            n2 - 1
        )
        *
        var2

    ) / (

        n1
        +
        n2
        -
        2
    )


    if pooled_variance <= 0:

        return 0.0


    cohen_d = (

        np.mean(x)
        -
        np.mean(y)

    ) / math.sqrt(
        pooled_variance
    )


    # Small-sample bias correction
    correction = (

        1.0
        -
        3.0
        /
        (
            4.0
            *
            (
                n1
                +
                n2
            )
            -
            9.0
        )
    )


    return float(
        correction
        *
        cohen_d
    )


def holm_adjust(
    p_values
):

    p_values = np.asarray(
        p_values,
        dtype=float
    )


    number_tests = len(
        p_values
    )


    order = np.argsort(
        p_values
    )


    adjusted = np.empty(
        number_tests,
        dtype=float
    )


    running_maximum = 0.0


    for rank, original_index in enumerate(
        order
    ):

        candidate = (

            (
                number_tests
                -
                rank
            )

            *
            p_values[
                original_index
            ]
        )


        running_maximum = max(

            running_maximum,
            candidate
        )


        adjusted[
            original_index
        ] = min(
            1.0,
            running_maximum
        )


    return adjusted


def effect_magnitude(
    g
):

    absolute = abs(
        g
    )


    if absolute < 0.20:

        return (
            "very small / negligible"
        )


    elif absolute < 0.50:

        return "small"


    elif absolute < 0.80:

        return "moderate"


    else:

        return "large"


# ============================================================
# 17. INFERENTIAL COMPARISON
#
# IMPORTANT:
# We do NOT run an independent t-test between:
#     Strict-fair subset (6759)
#     Full Clean unique (14689)
#
# because the former is contained in the latter.
#
# Instead:
#     Strict-fair paired = 6759
#     Excluded Clean     = 7930
#
# These two groups are mutually exclusive.
# ============================================================

inferential_rows = []

raw_p_values = []


for column, label in PRIMARY_METRICS:


    paired_values = paired_df[
        column
    ].to_numpy(
        dtype=float
    )


    excluded_values = excluded_df[
        column
    ].to_numpy(
        dtype=float
    )


    welch_result = stats.ttest_ind(

        paired_values,
        excluded_values,

        equal_var=False,
        nan_policy="omit"
    )


    (
        difference,
        ci_low,
        ci_high,
        welch_df

    ) = welch_mean_difference_ci(

        paired_values,
        excluded_values
    )


    g = hedges_g(

        paired_values,
        excluded_values
    )


    raw_p_values.append(
        float(
            welch_result.pvalue
        )
    )


    inferential_rows.append({

        "Feature":
            label,

        "Strict-fair mean":
            float(
                np.mean(
                    paired_values
                )
            ),

        "Excluded mean":
            float(
                np.mean(
                    excluded_values
                )
            ),

        "Delta paired-minus-excluded":
            difference,

        "95% CI low":
            ci_low,

        "95% CI high":
            ci_high,

        "Welch t":
            float(
                welch_result.statistic
            ),

        "Welch df":
            welch_df,

        "Raw p":
            float(
                welch_result.pvalue
            ),

        "Hedges g":
            g,

        "Effect magnitude":
            effect_magnitude(
                g
            ),
    })


adjusted_p_values = holm_adjust(
    raw_p_values
)


for row, adjusted_p in zip(

    inferential_rows,
    adjusted_p_values

):

    row[
        "Holm p"
    ] = float(
        adjusted_p
    )


    row[
        "Significant after Holm (alpha=.05)"
    ] = bool(
        adjusted_p < 0.05
    )


inferential_table = pd.DataFrame(
    inferential_rows
)


inferential_table = inferential_table[

    [
        "Feature",

        "Strict-fair mean",
        "Excluded mean",

        "Delta paired-minus-excluded",

        "95% CI low",
        "95% CI high",

        "Welch t",
        "Welch df",

        "Raw p",
        "Holm p",

        "Hedges g",
        "Effect magnitude",

        "Significant after Holm (alpha=.05)"
    ]
]


# ============================================================
# 18. PRINT TABLE A
# ============================================================

print(
    "\n\n"
    "============================================================"
)

print(
    "TABLE A. DESCRIPTIVE MORPHOLOGICAL REPRESENTATIVENESS"
)

print(
    "============================================================"
)


print(

    descriptive_table.to_string(
        index=False
    )
)


# ============================================================
# 19. PRINT TABLE B
# ============================================================

print(
    "\n\n"
    "============================================================"
)

print(
    "TABLE B. STRICT-FAIR PAIRED vs EXCLUDED CLEAN"
)

print(
    "============================================================"
)


print(
    "\nInferential comparison uses mutually exclusive "
    "unique-question groups:"
)


print(
    f"Strict-fair paired N = {len(paired_df):,}"
)

print(
    f"Excluded Clean N     = {len(excluded_df):,}\n"
)


print(

    inferential_table.to_string(
        index=False
    )
)


# ============================================================
# 20. DIRECTION SUMMARY
# ============================================================

print(
    "\n\n"
    "============================================================"
)

print(
    "DIRECTION SUMMARY"
)

print(
    "============================================================"
)


for _, row in inferential_table.iterrows():


    difference = row[
        "Delta paired-minus-excluded"
    ]


    if difference > 0:

        direction = "HIGHER"

    elif difference < 0:

        direction = "LOWER"

    else:

        direction = "EQUAL"


    print(

        f"\n• {row['Feature']}\n"
        f"  Strict-fair is {direction}\n"
        f"  Δ = {difference:.6f}\n"
        f"  95% CI = "
        f"[{row['95% CI low']:.6f}, "
        f"{row['95% CI high']:.6f}]\n"
        f"  Raw p = {row['Raw p']:.8g}\n"
        f"  Holm p = {row['Holm p']:.8g}\n"
        f"  Hedges' g = {row['Hedges g']:.6f}\n"
        f"  Effect = {row['Effect magnitude']}"
    )


# ============================================================
# 21. SAVE OUTPUTS
# ============================================================

if Path(
    "/content"
).exists():

    OUTPUT_DIRECTORY = Path(
        "/content/problem1_morphology_audit"
    )

else:

    OUTPUT_DIRECTORY = Path(
        "problem1_morphology_audit"
    )


OUTPUT_DIRECTORY.mkdir(

    parents=True,
    exist_ok=True
)


descriptive_path = (

    OUTPUT_DIRECTORY
    /
    "problem1_descriptive_representativeness.csv"
)


inferential_path = (

    OUTPUT_DIRECTORY
    /
    "problem1_inferential_paired_vs_excluded.csv"
)


item_level_path = (

    OUTPUT_DIRECTORY
    /
    "problem1_unique_question_level_features.csv"
)


validation_path = (

    OUTPUT_DIRECTORY
    /
    "problem1_segmenter_validation.txt"
)


descriptive_table.to_csv(

    descriptive_path,

    index=False,
    encoding="utf-8-sig"
)


inferential_table.to_csv(

    inferential_path,

    index=False,
    encoding="utf-8-sig"
)


unique_df.to_csv(

    item_level_path,

    index=False,
    encoding="utf-8-sig"
)


with open(

    validation_path,
    "w",
    encoding="utf-8"

) as file:


    file.write(

        f"Exact normalized segmentation match: "
        f"{exact_match}/{n_common} "
        f"= {exact_rate:.8%}\n"
    )


    file.write(

        f"Boundary-count match: "
        f"{boundary_match}/{n_common} "
        f"= {boundary_rate:.8%}\n"
    )


    file.write(
        "\nPAIRING COUNTS\n"
    )


    for key, value in ACTUAL.items():

        file.write(
            f"{key}: {value}\n"
        )


    file.write(
        "\nRESOURCE COUNTS\n"
    )


    for key, value in ACTUAL_RESOURCES.items():

        file.write(
            f"{key}: {value}\n"
        )


# ============================================================
# 22. ZIP OUTPUTS
# ============================================================

zip_path = (

    OUTPUT_DIRECTORY.parent
    /
    "problem1_morphology_audit_outputs.zip"
)


with zipfile.ZipFile(

    zip_path,
    "w",
    zipfile.ZIP_DEFLATED

) as archive:


    for path in [

        descriptive_path,
        inferential_path,
        item_level_path,
        validation_path

    ]:

        archive.write(

            path,

            arcname=path.name
        )


print(
    "\n\n"
    "============================================================"
)

print(
    "FILES SAVED"
)

print(
    "============================================================"
)


print(
    descriptive_path
)

print(
    inferential_path
)

print(
    item_level_path
)

print(
    validation_path
)

print(
    zip_path
)


# ============================================================
# 23. SPECIAL BLOCK TO SEND BACK TO CHATGPT
# ============================================================

print(
    "\n\n"
    "============================================================"
)

print(
    "COPY THIS BLOCK BACK TO CHATGPT"
)

print(
    "============================================================"
)


print(

    f"\nClean records = {len(base_rows)}"

    f"\nClean unique = {len(base_map)}"

    f"\nSegmented records = {len(seg_rows)}"

    f"\nSegmented unique = {len(seg_map)}"

    f"\nStrict-fair paired = {len(common_keys)}"

    f"\nExcluded Clean = {len(excluded_keys)}"
)


print(

    f"\nSegmenter exact-match "
    f"= {exact_rate:.4%}"

    f"\nBoundary-count match "
    f"= {boundary_rate:.4%}"
)


print(
    "\n\nDESCRIPTIVE RESULTS:"
)


print(

    descriptive_table[
        [
            "Feature",

            "Full Clean all mean",

            "Full Clean unique mean",

            "Strict-fair mean",

            "Excluded unique mean",

            "Strict-fair vs Full-unique relative diff %"
        ]
    ].to_string(
        index=False
    )
)


print(
    "\n\nINFERENTIAL RESULTS:"
)


print(

    inferential_table[
        [
            "Feature",

            "Delta paired-minus-excluded",

            "95% CI low",
            "95% CI high",

            "Raw p",
            "Holm p",

            "Hedges g",

            "Effect magnitude"
        ]
    ].to_string(
        index=False
    )
)


print(
    "\n\n✅ DONE."
)

print(
    "Маған жоғарыдағы 'COPY THIS BLOCK BACK TO CHATGPT' "
    "бөлігін толық көшіріп жіберіңіз."
)

print(
    "Содан кейін нақты мәндерге сүйеніп "
    "Reviewer 1/problem 1 → Reply 1 және "
    "мақалаға кіретін дайын кесте/мәтінді жасаймыз."
)


# ============================================================
# OPTIONAL:
# ZIP файлын автоматты жүктеу керек болса,
# төмендегі екі жолдың алдындағы # белгілерін алып тастаңыз.
# ============================================================

# if IN_COLAB:
#     files.download(str(zip_path))


================ FILES USED ================
base      : /content/baseline_15000.json
seg       : /content/kazakh_segmented_15000.json
stems     : /content/qaz_stems_unik_edit.xlsx
jurnaqs   : /content/qaz_jurnaks.xls
endings   : /content/qaz_endings_seg.xls
stop      : /content/stop_words.txt

================ PAIRING AUDIT ================
Clean records                  = 14,991
Segmented records              = 14,998
Clean unique normalized keys   = 14,689
Segmented unique norm. keys    = 14,696
Strict-fair common keys        = 6,759
Excluded Clean unique keys     = 7,930
Clean duplicate rows collapsed = 302
Seg duplicate rows collapsed   = 302

✅ Pairing counts EXACTLY match the manuscript.
✅ stop_words encoding=utf-8 | non-empty=189

================ RESOURCE AUDIT ================
Stem entries                    = 103,624
Unique stem forms               = 103,623
Derivational-suffix entries     = 146
Unique derivational suffixes    = 101
Ending→segmentation mappings    = 3,316
U